# Speed Limit Sign Detection - YOLOv8 Training (Google Drive)

This notebook trains a YOLOv8-nano model to detect speed limit signs (25-80 mph).

**Dataset**: 972 labeled images from Google Drive  
**Model**: YOLOv8-nano (optimized for Raspberry Pi)  
**Expected training time**: ~10-15 minutes on Colab T4 GPU

## Prerequisites

### ✅ Before running this notebook:

1. **Upload dataset to Google Drive:**
   - Create folder: `My Drive/CarCam/dataset/`
   - Upload the entire dataset folder structure:
     ```
     My Drive/
     └── CarCam/
         └── dataset/
             ├── images/
             │   ├── train/  (871 PNG files)
             │   └── val/    (101 PNG files)
             ├── labels/
             │   ├── train/  (871 TXT files)
             │   └── val/    (100 TXT files)
             └── dataset.yaml
     ```

2. **Enable GPU in Colab:**
   - Runtime → Change runtime type → T4 GPU

3. **Run all cells in order**

---

## 1. Check GPU Availability

In [ ]:
!nvidia-smi

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Verify dataset exists
DATASET_PATH = '/content/drive/MyDrive/CarCam/dataset'

if os.path.exists(DATASET_PATH):
    print(f"✅ Dataset found at: {DATASET_PATH}")
    print(f"\nContents:")
    !ls -lh /content/drive/MyDrive/CarCam/dataset/
else:
    print(f"❌ ERROR: Dataset not found at {DATASET_PATH}")
    print(f"\nPlease upload your dataset to Google Drive:")
    print(f"  1. Go to Google Drive")
    print(f"  2. Create folder: My Drive/CarCam/dataset/")
    print(f"  3. Upload the dataset/ folder from your local machine")
    raise FileNotFoundError(f"Dataset not found at {DATASET_PATH}")

## 3. Install Dependencies

In [ ]:
!pip install -q ultralytics

from ultralytics import YOLO
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 4. Verify Dataset Structure

In [ ]:
import os
from pathlib import Path

# Check dataset structure
train_images = list(Path(f"{DATASET_PATH}/images/train").glob("*.png"))
val_images = list(Path(f"{DATASET_PATH}/images/val").glob("*.png"))
train_labels = list(Path(f"{DATASET_PATH}/labels/train").glob("*.txt"))
val_labels = list(Path(f"{DATASET_PATH}/labels/val").glob("*.txt"))

print("Dataset Statistics:")
print(f"  Train images: {len(train_images)}")
print(f"  Train labels: {len(train_labels)}")
print(f"  Val images: {len(val_images)}")
print(f"  Val labels: {len(val_labels)}")
print(f"  Total: {len(train_images) + len(val_images)} images")

# Verify dataset.yaml exists
dataset_yaml = f"{DATASET_PATH}/dataset.yaml"
if os.path.exists(dataset_yaml):
    print(f"\n✅ dataset.yaml found")
    print(f"\nContents:")
    !cat {dataset_yaml}
else:
    print(f"\n❌ dataset.yaml not found")
    raise FileNotFoundError("Missing dataset.yaml")

# Display sample image
from PIL import Image
import matplotlib.pyplot as plt

sample_img = Image.open(train_images[0])
plt.figure(figsize=(10, 6))
plt.imshow(sample_img)
plt.title(f"Sample image: {train_images[0].name}")
plt.axis('off')
plt.show()

print(f"\nImage size: {sample_img.size}")

## 5. Update dataset.yaml Path

Since the dataset is on Google Drive, we need to update the path in dataset.yaml

In [ ]:
import yaml

# Read dataset.yaml
with open(dataset_yaml, 'r') as f:
    dataset_config = yaml.safe_load(f)

# Update path to point to Google Drive
dataset_config['path'] = DATASET_PATH

# Write temporary dataset.yaml to Colab
temp_yaml = '/content/dataset.yaml'
with open(temp_yaml, 'w') as f:
    yaml.dump(dataset_config, f)

print("Updated dataset.yaml for Colab:")
!cat {temp_yaml}

DATASET_YAML_PATH = temp_yaml

## 6. Configure Training Parameters

In [ ]:
# Training configuration
config = {
    'data': DATASET_YAML_PATH,
    'epochs': 100,
    'imgsz': 640,  # Input image size (resized from 1920x1080)
    'batch': 16,   # Batch size (adjust based on GPU memory)
    'patience': 20,  # Early stopping patience
    'device': 0,   # GPU device
    'workers': 2,
    'project': 'runs/train',
    'name': 'speed_limit_detector',
    'exist_ok': True,
    
    # Small object detection optimizations
    'mosaic': 1.0,  # Mosaic augmentation
    'copy_paste': 0.5,  # Copy-paste augmentation
    'degrees': 0.0,  # No rotation (signs are always upright)
    'translate': 0.1,  # Slight translation
    'scale': 0.5,  # Scale augmentation for size variation
    'fliplr': 0.5,  # Horizontal flip
    'flipud': 0.0,  # No vertical flip (signs don't appear upside down)
    'hsv_h': 0.015,  # HSV-Hue augmentation
    'hsv_s': 0.7,  # HSV-Saturation
    'hsv_v': 0.4,  # HSV-Value
}

print("Training Configuration:")
for key, value in config.items():
    print(f"  {key}: {value}")

## 7. Train YOLOv8-Nano Model

In [ ]:
# Initialize YOLOv8-nano model
model = YOLO('yolov8n.pt')  # Load pretrained nano model

# Train the model
print("\nStarting training...")
print("This will take approximately 10-15 minutes on T4 GPU\n")

results = model.train(**config)

print("\n✅ Training complete!")

## 8. Evaluate Model Performance

In [ ]:
# Run validation
metrics = model.val()

print("\nValidation Metrics:")
print(f"  mAP50: {metrics.box.map50:.4f}")
print(f"  mAP50-95: {metrics.box.map:.4f}")
print(f"  Precision: {metrics.box.mp:.4f}")
print(f"  Recall: {metrics.box.mr:.4f}")

## 9. Visualize Training Results

In [ ]:
from IPython.display import Image as IPImage, display

# Display training curves
results_path = f"{config['project']}/{config['name']}"

print("Training Results:")
display(IPImage(filename=f"{results_path}/results.png"))

print("\nConfusion Matrix:")
display(IPImage(filename=f"{results_path}/confusion_matrix.png"))

## 10. Test on Validation Images

In [ ]:
# Load best model
best_model = YOLO(f"{results_path}/weights/best.pt")

# Run inference on sample validation images
val_sample = str(val_images[0])
results = best_model.predict(val_sample, conf=0.25, save=True)

print(f"Sample prediction saved to: {results[0].save_dir}")

# Display prediction
for r in results:
    im_array = r.plot()  # plot a BGR numpy array of predictions
    im = Image.fromarray(im_array[..., ::-1])  # RGB PIL image
    plt.figure(figsize=(12, 8))
    plt.imshow(im)
    plt.axis('off')
    plt.title("Sample Prediction")
    plt.show()

## 11. Export Model for Raspberry Pi

In [ ]:
# Export to ONNX format (best for Raspberry Pi)
print("Exporting to ONNX format...")
onnx_path = best_model.export(format='onnx', dynamic=False, simplify=True)
print(f"\n✅ Model exported to ONNX: {onnx_path}")

# Copy models to a download directory
import shutil
!mkdir -p /content/trained_models

shutil.copy(f"{results_path}/weights/best.pt", "/content/trained_models/best.pt")
shutil.copy(onnx_path, "/content/trained_models/best.onnx")

print("\nModels copied to /content/trained_models/")
print("You can also save models back to Google Drive (see next cell)")

## 12. Save Models to Google Drive (Optional)

In [ ]:
# Save trained models back to Google Drive for safekeeping
drive_models_path = '/content/drive/MyDrive/CarCam/models'
!mkdir -p {drive_models_path}

shutil.copy("/content/trained_models/best.pt", f"{drive_models_path}/best.pt")
shutil.copy("/content/trained_models/best.onnx", f"{drive_models_path}/best.onnx")

print(f"✅ Models saved to Google Drive: {drive_models_path}")
print("\nYou can access these models anytime from your Google Drive!")

## 13. Download Trained Models

In [ ]:
from google.colab import files

# Download PyTorch model
print("Downloading models...")
files.download("/content/trained_models/best.pt")

# Download ONNX model (for Raspberry Pi)
files.download("/content/trained_models/best.onnx")

print("\n✅ Models downloaded!")
print("\nNext steps:")
print("1. Copy best.onnx to your Raspberry Pi: ~/CarCam/models/")
print("2. Clone GitHub repo on Pi: git clone https://github.com/yoshimoshi/CarCam.git")
print("3. Install dependencies: pip install -r app/requirements.txt")
print("4. Run detection: python app/detect.py")

## 14. Model Summary

In [ ]:
import os

# Get model file sizes
pt_size = os.path.getsize("/content/trained_models/best.pt") / (1024**2)
onnx_size = os.path.getsize("/content/trained_models/best.onnx") / (1024**2)

print("="*60)
print("TRAINING SUMMARY")
print("="*60)
print(f"\nModel Architecture: YOLOv8-nano")
print(f"Training Dataset: 871 images")
print(f"Validation Dataset: {len(val_images)} images")
print(f"Number of Classes: 12")
print(f"Input Size: 640x640")
print(f"\nModel Performance:")
print(f"  mAP50: {metrics.box.map50:.4f}")
print(f"  mAP50-95: {metrics.box.map:.4f}")
print(f"  Precision: {metrics.box.mp:.4f}")
print(f"  Recall: {metrics.box.mr:.4f}")
print(f"\nModel Files:")
print(f"  PyTorch (.pt): {pt_size:.2f} MB")
print(f"  ONNX (.onnx): {onnx_size:.2f} MB")
print(f"\nExpected Inference Speed:")
print(f"  Colab T4 GPU: ~5-10ms per frame")
print(f"  Raspberry Pi 4 CPU: ~100-200ms per frame")
print("\n" + "="*60)
print("Training complete! Models ready for deployment.")
print("="*60)